# Registers — Flip-Flops in a Row

A register is a bank of flip-flops sharing one clock, storing or moving a multi-bit word. This notebook draws the **cell chain as a circuit block** and shows the **bit physically marching through the stages** on each clock edge, the defining behaviour of a shift register.

$$Q_i^{next} = Q_{i-1}\big|_{CLK\uparrow} \quad\text{(right shift)}$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrow
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 9,
})

ON, OFF = '#c0392b', '#b0b0b0'
def wcol(b): return ON if b else OFF
def wlw(b):  return 2.4 if b else 1.2

def clock(n, period=4):
    t = np.arange(n)
    return ((t // (period//2)) % 2).astype(int)

def rising_edges(clk):
    return [i for i in range(1, len(clk)) if clk[i-1]==0 and clk[i]==1]

def ff_cell(ax, x, y, bit, label, w=0.95, h=0.9):
    """Draw one register cell (a D flip-flop) lit by its stored bit."""
    fc = '#fdecea' if bit else '#eef2f7'
    ax.add_patch(Rectangle((x, y-h/2), w, h, fc=fc, ec='#34495e', lw=1.5, zorder=2))
    ax.text(x+w/2, y+0.12, label, ha='center', va='center', fontsize=8, color='#555', zorder=3)
    ax.text(x+w/2, y-0.16, str(bit), ha='center', va='center', fontsize=13,
            weight='bold', color=wcol(bit), zorder=3)
    ax.plot([x, x+0.18, x],[y-h/2+0.1, y-h/2+0.22, y-h/2+0.34], color='#34495e', lw=1.1, zorder=3)
    return (x, y), (x+w, y)

print('primitives ready')


## One Cell — A 1-Bit Register

The atom of every register is a single D flip-flop with a parallel load. On each rising edge it latches the input bit; between edges it holds. An $N$-bit register is just $N$ of these clocked together.


In [ ]:
def one_cell(D_pat, period):
    n = 24
    base = [int(c) for c in D_pat.ljust(6,'0')[:6]]
    D = np.array([base[i % len(base)] for i in range(n)])
    CLK = clock(n, period); edges = rising_edges(CLK)
    Q = np.zeros(n, dtype=int); q = 0
    for i in range(n):
        if i in edges: q = D[i]
        Q[i] = q
    t = np.arange(n)
    fig, axes = plt.subplots(3,1, figsize=(8,3.4), sharex=True)
    for ax, sig, lab, c in zip(axes, [CLK, D, Q], ['CLK','D','Q'], ['#34495e','#2471a3','#c0392b']):
        ax.step(t, sig, where='post', color=c, lw=2)
        ax.set_ylim(-0.3,1.3); ax.set_yticks([0,1])
        ax.set_ylabel(lab, rotation=0, ha='right', va='center'); ax.grid(True, alpha=0.3)
        for e in edges: ax.axvline(e, color='#8e44ad', ls=':', lw=0.8, alpha=0.6)
    axes[-1].set_xlabel('time tick')
    plt.tight_layout(); plt.show()

w_d1 = widgets.Text(value='101100', description='D cycle:', layout=widgets.Layout(width='400px'))
w_p1 = widgets.IntSlider(value=4, min=2, max=6, step=2, description='period:')
display(widgets.VBox([w_d1, w_p1]),
        widgets.interactive_output(one_cell, {'D_pat': w_d1, 'period': w_p1}))


## Shift Register — Step the Schematic Through Time

Each cell's output drives the next cell's input, so contents **shift one stage per clock edge**. Move the **clock-pulse slider**: the cell chain redraws with the lit bit marching left-to-right, the clock waveform below marks the current edge with a pointer, and a heatmap shows the full evolution. This is the schematic *and* its temporal evolution together.

$$Q_3 Q_2 Q_1 Q_0 \;\xrightarrow{CLK\uparrow}\; \text{in},Q_3,Q_2,Q_1$$


In [ ]:
def shift_register(serial_in, n_clocks):
    N = 4
    stream = [int(c) for c in serial_in.ljust(8,'0')[:8]]
    history = [[0]*N]
    reg = [0]*N
    for k in range(8):
        sin = stream[k % len(stream)]
        reg = [sin] + reg[:-1]
        history.append(reg[:])
    cur = history[n_clocks]
    fig = plt.figure(figsize=(8.8, 5.2))
    # --- schematic (top) ---
    ax1 = fig.add_axes([0.06, 0.62, 0.88, 0.34]); ax1.axis('off')
    ax1.set_xlim(0, N+1.6); ax1.set_ylim(0, 2)
    sin_now = stream[(n_clocks-1) % len(stream)] if n_clocks>0 else stream[0]
    ax1.annotate('', xy=(0.55, 1.0), xytext=(0.0, 1.0),
                 arrowprops=dict(arrowstyle='->', color=wcol(sin_now), lw=wlw(sin_now)))
    ax1.text(0.0, 1.35, f'serial in={sin_now}', color=wcol(sin_now), fontsize=9, weight='bold')
    prev_out = None
    for i in range(N):
        x = 0.7 + i*1.0
        pin_a, pout = ff_cell(ax1, x, 1.0, cur[i], f'D-FF Q{N-1-i}')
        if prev_out is not None:
            ax1.annotate('', xy=(x, 1.0), xytext=prev_out,
                         arrowprops=dict(arrowstyle='->', color='#888', lw=1.2))
        prev_out = pout
    ax1.annotate('', xy=(N+1.2,1.0), xytext=prev_out, arrowprops=dict(arrowstyle='->',color='#888',lw=1.2))
    ax1.text(N+1.2,1.35,'serial out',fontsize=8,color='#555')
    ax1.text((N)/2+0.3, 0.25, f'after clock pulse {n_clocks}: {cur}', ha='center', fontsize=9, color='#555')
    # --- clock waveform with pointer (middle) ---
    axc = fig.add_axes([0.06, 0.40, 0.88, 0.16])
    period=4; n_t=(8)*period+2; t=np.arange(n_t); CLK=clock(n_t,period); edges=rising_edges(CLK)
    axc.step(t,CLK,where='post',color='#34495e',lw=1.6)
    axc.set_ylim(-0.3,1.3); axc.set_yticks([0,1]); axc.set_ylabel('CLK',rotation=0,ha='right'); axc.grid(True,alpha=0.3)
    for k,e in enumerate(edges[:9]): axc.axvline(e,color='#8e44ad',ls=':',lw=0.8,alpha=0.5)
    if n_clocks>0 and n_clocks<=len(edges): axc.axvline(edges[n_clocks-1],color='#8e44ad',lw=2.5)
    axc.set_xlabel('time  (bold = current edge)')
    # --- heatmap (bottom) ---
    ax2 = fig.add_axes([0.06, 0.08, 0.88, 0.22])
    H = np.array(history).T
    ax2.imshow(H, aspect='auto', cmap='Reds', vmin=0, vmax=1, interpolation='nearest')
    ax2.set_yticks(range(N)); ax2.set_yticklabels([f'Q{N-1-i}' for i in range(N)])
    ax2.set_xlabel('clock pulse'); ax2.axvline(n_clocks, color='#8e44ad', lw=2)
    ax2.set_title('register contents over time', fontsize=9)
    plt.show()

w_si = widgets.Text(value='10110010', description='serial in:', layout=widgets.Layout(width='400px'))
w_nc = widgets.IntSlider(value=4, min=0, max=8, description='clock pulse:')
display(widgets.VBox([w_si, w_nc]),
        widgets.interactive_output(shift_register, {'serial_in': w_si, 'n_clocks': w_nc}))


## Four Modes — SISO, SIPO, PISO, PIPO

Registers differ in how data enters and leaves: **S**erial or **P**arallel **I**n, **S**erial or **P**arallel **O**ut. SIPO is a serial-to-parallel converter; PISO is the reverse. The table shows the latency and use of each.

| Mode | In | Out | Clocks to fill/read | Typical use |
|------|----|----|----|----|
| SISO | serial | serial | N to traverse | delay line |
| SIPO | serial | parallel | N to fill | serial->parallel RX |
| PISO | parallel | serial | 1 load + N out | parallel->serial TX |
| PIPO | parallel | parallel | 1 | buffer / storage |


In [ ]:
def sipo_demo(serial_in):
    N = 4
    stream = [int(c) for c in serial_in.ljust(N,'0')[:N]]
    reg = [0]*N; snapshots = [reg[:]]
    for k in range(N):
        reg = [stream[k]] + reg[:-1]; snapshots.append(reg[:])
    print('SIPO: feeding', stream, 'serially, reading all bits in parallel after', N, 'clocks')
    for k, s in enumerate(snapshots):
        tag = '  <- parallel read' if k == N else ''
        print(f'  after clock {k}: {s}{tag}')
    # PISO: load parallel, shift out serial
    load = [int(c) for c in serial_in.ljust(N,'0')[:N]]
    reg = load[:]; out = []
    for k in range(N):
        out.append(reg[-1]); reg = [0] + reg[:-1]
    print(f'\nPISO: parallel load {load} -> serial out stream {out}')

w_sp = widgets.Text(value='1011', description='word:', layout=widgets.Layout(width='300px'))
display(w_sp, widgets.interactive_output(sipo_demo, {'serial_in': w_sp}))


## Universal Shift Register — Direction Control

A direction line selects whether the chain shifts right or left at each edge; a mode can also hold or parallel-load. The heatmap shows the same word shifting one way, then the other, as you flip the direction.


In [ ]:
def universal_shift(initial, dir_pattern):
    N = 5
    reg = [int(c) for c in initial.ljust(N,'0')[:N]]
    dirs = [int(c) for c in dir_pattern.ljust(10,'1')[:10]]  # 1=right, 0=left
    hist = [reg[:]]
    for d in dirs:
        if d: reg = [0] + reg[:-1]      # shift right
        else: reg = reg[1:] + [0]       # shift left
        hist.append(reg[:])
    H = np.array(hist).T
    fig, ax = plt.subplots(figsize=(8.5,2.8))
    ax.imshow(H, aspect='auto', cmap='Reds', vmin=0, vmax=1, interpolation='nearest')
    ax.set_yticks(range(N)); ax.set_yticklabels([f'Q{N-1-i}' for i in range(N)])
    ax.set_xlabel('clock pulse')
    for k, d in enumerate(dirs):
        ax.text(k+1, -0.7, 'R' if d else 'L', ha='center', fontsize=8,
                color='#2471a3' if d else '#e67e22')
    ax.set_title('R = shift right, L = shift left', fontsize=9)
    plt.tight_layout(); plt.show()

w_init = widgets.Text(value='11000', description='initial:', layout=widgets.Layout(width='320px'))
w_dir = widgets.Text(value='1111100000', description='dir (1=R,0=L):', layout=widgets.Layout(width='420px'))
display(widgets.VBox([w_init, w_dir]),
        widgets.interactive_output(universal_shift, {'initial': w_init, 'dir_pattern': w_dir}))


## LFSR — Feedback Turns a Shift Register Into a Sequence Generator

Tapping certain stages, XOR-ing them, and feeding the result back as the serial input produces a long pseudo-random cycle. A maximal $n$-bit LFSR visits all $2^n - 1$ non-zero states before repeating, the basis of CRC and PRNG hardware.

$$\text{feedback} = \bigoplus_{i \in \text{taps}} Q_i$$


In [ ]:
def lfsr_run(seed, taps_str, n_bits):
    taps = [int(x) for x in taps_str.split(',') if x.strip().isdigit()]
    state = seed & ((1 << n_bits) - 1)
    if state == 0: state = 1
    seen = []; s = state
    for _ in range(2**n_bits + 2):
        if s in seen: break
        seen.append(s)
        fb = 0
        for tp in taps:
            fb ^= (s >> tp) & 1
        s = ((s >> 1) | (fb << (n_bits-1))) & ((1 << n_bits)-1)
    period = len(seen)
    maximal = period == (2**n_bits - 1)
    fig, ax = plt.subplots(figsize=(8.5,3))
    M = np.array([[(v >> b) & 1 for b in range(n_bits)] for v in seen]).T
    ax.imshow(M, aspect='auto', cmap='Reds', vmin=0, vmax=1, interpolation='nearest')
    ax.set_yticks(range(n_bits)); ax.set_yticklabels([f'Q{b}' for b in range(n_bits)])
    ax.set_xlabel('step')
    ax.set_title(f'period = {period}  (max possible = {2**n_bits-1})  '
                 f'-> {"MAXIMAL" if maximal else "not maximal"}',
                 color='#2ca02c' if maximal else '#c0392b', fontsize=9.5)
    plt.tight_layout(); plt.show()
    print('state sequence:', [format(v, f'0{n_bits}b') for v in seen[:16]],
          '...' if period > 16 else '')

w_seed = widgets.IntSlider(value=1, min=1, max=15, description='seed:')
w_taps = widgets.Text(value='0,1', description='taps (idx):', layout=widgets.Layout(width='320px'))
w_nb = widgets.IntSlider(value=4, min=3, max=5, description='bits:')
display(widgets.VBox([w_seed, w_taps, w_nb]),
        widgets.interactive_output(lfsr_run, {'seed': w_seed, 'taps_str': w_taps, 'n_bits': w_nb}))
